# AirLLM 4.0 benchmark on Kaggle T4

Gemma 4 (E2B-qat, E4B, 12B, 26B-A4B) and Qwen 3.5-9B / 3.6-27B / 3.8-27B, streamed layer by layer with AirLLM. Generated from `scripts/make_kaggle_notebook.py` in the airllm-benchmarks repo.

In [ ]:
!nvidia-smi
!df -h / /kaggle/working /tmp
!free -g
!nproc

In [ ]:
%%capture pip_log
!pip install -q 'airllm==4.0.0' 'transformers>=5.8,<6' accelerate safetensors bitsandbytes compressed-tensors 'huggingface_hub[hf_xet]' psutil nvidia-ml-py sentencepiece

In [ ]:
import torch, transformers, importlib.metadata as md
print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.get_device_name(0))
print('transformers', transformers.__version__, 'airllm', md.version('airllm'), 'bnb', md.version('bitsandbytes'), 'ct', md.version('compressed-tensors'))
print('bf16 native:', torch.cuda.is_bf16_supported(including_emulation=False))

In [ ]:
import os
os.makedirs('/kaggle/working/airllm-benchmarks/bench', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/airllm-benchmarks/bench/airllm_compat.py
"""Compatibility shims for AirLLM 4.0.0 on newer transformers (tested: 5.23).

AirLLM builds a pre-quantized model's quantizer with ``AutoHfQuantizer.from_config`` and
calls ``preprocess_model`` straight away. transformers' own loader calls
``validate_environment()`` in between. Recent versions set required state there (e.g.
``CompressedTensorsHfQuantizer.use_fp8_kernel``), so without that call loading any
compressed-tensors checkpoint fails with an AttributeError.
"""

import airllm.airllm_base as _base
from transformers.quantizers.auto import AutoHfQuantizer as _AutoHfQuantizer


class _AutoHfQuantizerShim:
    @staticmethod
    def from_config(*args, **kwargs):
        quantizer = _AutoHfQuantizer.from_config(*args, **kwargs)
        try:
            quantizer.validate_environment(device_map=None)
        except TypeError:
            quantizer.validate_environment()
        return quantizer


_base.AutoHfQuantizer = _AutoHfQuantizerShim


# --- compressed-tensors checkpoints with activation quantization (e.g. Gemma 4 QAT "-ct") -------
#
# 1. ``AirLLMBaseModel._decompress_state_dict`` expands ``weight_packed`` into ``weight`` but
#    pops *every* tensor under that module's prefix, including the static activation scales
#    (``input_scale``) that compressed-tensors' quantized_forward needs for W4A8 / W2A8 layers.
#    Those stay on the meta device, and the first forward crashes.
# 2. ``int-quantized`` (plain int8) weights are not packed, so AirLLM loads them verbatim as int8
#    into a module whose forward does a bf16 matmul. Dequantize them (symmetric, per channel or group).

import torch as _torch

_ACT_SUFFIXES = (".input_scale", ".input_zero_point", ".output_scale", ".output_zero_point",
                 ".input_global_scale")
_orig_decompress = _base.AirLLMBaseModel._decompress_state_dict


def _has_tensor(module, attr):
    return attr in getattr(module, "_parameters", {}) or attr in getattr(module, "_buffers", {})


def _decompress_state_dict_fixed(self, state_dict):
    out = _orig_decompress(self, state_dict)
    if self.hf_quantizer is None:
        return out
    if out is state_dict:
        out = dict(state_dict)

    for key, value in state_dict.items():
        if not key.endswith(_ACT_SUFFIXES):
            continue
        prefix, _, attr = key.rpartition(".")
        try:
            module = self.model.get_submodule(prefix)
        except AttributeError:
            continue
        if _has_tensor(module, attr):
            # The empty model is built in fp32 and accelerate casts loaded values to the existing
            # parameter's dtype, so an fp32 input_scale would promote the fake-quantized
            # activation to fp32 and break the bf16 matmul that follows. Re-type the meta param.
            if value.is_floating_point() and attr in module._parameters:
                old = module._parameters[attr]
                module._parameters[attr] = _torch.nn.Parameter(
                    _torch.empty(old.shape, device="meta", dtype=self.running_dtype), requires_grad=False)
                value = value.to(self.running_dtype)
            out[key] = value

    for key in [k for k in out if k.endswith(".weight") and not out[k].is_floating_point()]:
        prefix = key[: -len(".weight")]
        scale = out.get(prefix + ".weight_scale")
        if scale is None:
            continue
        try:
            module = self.model.get_submodule(prefix)
        except AttributeError:
            continue
        q = out[key].to(self.running_device, _torch.float32)
        s = scale.to(self.running_device, _torch.float32)
        if s.dim() == 2 and s.shape[1] not in (1, q.shape[1]):  # group-wise
            g = q.shape[1] // s.shape[1]
            w = (q.view(q.shape[0], s.shape[1], g) * s[:, :, None]).view_as(q)
        else:
            w = q * s
        zp = out.pop(prefix + ".weight_zero_point", None)
        if zp is not None and zp.abs().sum() > 0:
            raise NotImplementedError(f"asymmetric int weights not handled: {prefix}")
        out[key] = w.to(self.running_dtype)
        out.pop(prefix + ".weight_scale", None)
        self._expose_plain_weight(module, out[key])
    return out


_base.AirLLMBaseModel._decompress_state_dict = _decompress_state_dict_fixed


# --- optional: NF4-compress only weight matrices ----------------------------------------------
#
# AirLLM's ``compression='4bit'`` quantizes *every* tensor of a layer to NF4 (blocksize 64),
# including RMSNorm weights, per-layer scalars and other 1-D parameters. Regular weight-only
# quantizers (bitsandbytes / GPTQ / AWQ) leave those in full precision. With
# AIRLLM_COMPRESS_MIN_DIM=2 only tensors with >= 2 dims are compressed; the rest are stored as is.

import os as _os

import airllm.utils as _utils

_COMPRESS_MIN_DIM = int(_os.environ.get("AIRLLM_COMPRESS_MIN_DIM", "0"))
_orig_compress = _utils.compress_layer_state_dict
_orig_uncompress = _utils.uncompress_layer_state_dict


def _compress_min_dim(layer_state_dict, compression=None):
    if compression is None or _COMPRESS_MIN_DIM <= 1:
        return _orig_compress(layer_state_dict, compression)
    big = {k: v for k, v in layer_state_dict.items() if v.dim() >= _COMPRESS_MIN_DIM}
    out = _orig_compress(big, compression) if big else {}
    out.update({k: v for k, v in layer_state_dict.items() if v.dim() < _COMPRESS_MIN_DIM})
    return out


def _uncompress_mixed(layer_state_dict):
    """Original only handles all-or-nothing compressed shards; pass uncompressed tensors through."""
    tags = (".4bit.", ".8bit.")
    if not any(t in k for k in layer_state_dict for t in tags):
        return layer_state_dict
    bases = {k.split(".4bit.")[0].split(".8bit.")[0] for k in layer_state_dict if any(t in k for t in tags)}
    quant = {k: v for k, v in layer_state_dict.items() if k in bases or any(t in k for t in tags)}
    out = _orig_uncompress(quant)
    out.update({k: v for k, v in layer_state_dict.items() if k not in quant})
    return out


_utils.compress_layer_state_dict = _compress_min_dim
_utils.uncompress_layer_state_dict = _uncompress_mixed


# --- optional: skip AirLLM's per-layer garbage collection -------------------------------------
#
# AirLLM calls clean_memory() (gc.collect + malloc_trim + torch.cuda.empty_cache) after *every*
# streamed module. With a transformers model graph alive, gc.collect() alone costs ~100 ms
# (≈435k tracked objects for Gemma 4 E4B), i.e. several seconds per generated token. Tensors are
# freed by reference counting anyway. AIRLLM_NO_GC=1 turns the per-layer call into a no-op.

if _os.environ.get("AIRLLM_NO_GC") == "1":
    _base.clean_memory = lambda: None


In [ ]:
%%writefile /kaggle/working/airllm-benchmarks/bench/airllm_gemma4.py
"""AirLLM adapter for Google Gemma 4 checkpoints.

AirLLM 4.0.0 has no Gemma 4 class. Its generic ``AirLLMBaseModel`` looks for
``model.embed_tokens`` / ``model.layers``, which do not exist in
``Gemma4ForConditionalGeneration`` (or ``Gemma4UnifiedForConditionalGeneration``), so
``AutoModel.from_pretrained`` fails while building the empty model.

This adapter maps the text decoder under ``model.language_model`` and handles the
part specific to Gemma 4's "E" models (E2B / E4B): the Per-Layer Embedding (PLE) table
``embed_tokens_per_layer``. It is a [262144 x num_layers*256] lookup table (5.6 GB bf16
for E4B). Streaming it to the GPU for every token would cost more than all the decoder
layers together. Only a handful of rows are needed per step, so we keep it in host RAM
and gather rows on the CPU. That is the same idea AirLLM itself uses for Qwen3.8
Flash-Next's n-gram table.

Text-only: vision / audio towers are not split or loaded.
"""

from __future__ import annotations

import types

import torch
import torch.nn.functional as F

import airllm.utils as airllm_utils
from airllm import AirLLMBaseModel

PLE_NAME = "model.language_model.embed_tokens_per_layer"

# The PLE table is a CPU lookup table that never streams, so 4/8-bit block compression
# gains nothing there and would only break the CPU gather. Skip it when splitting.
_original_compress = airllm_utils.compress_layer_state_dict


def _compress_skip_ple(layer_state_dict, compression=None):
    if compression is not None and any(PLE_NAME in k for k in layer_state_dict):
        return layer_state_dict
    return _original_compress(layer_state_dict, compression)


airllm_utils.compress_layer_state_dict = _compress_skip_ple


def _ple_cpu_forward(module, input_ids):
    table = module._airllm_cpu_table
    out = F.embedding(input_ids.to("cpu"), table, module.padding_idx)
    out = out.to(input_ids.device, non_blocking=True)
    return out * module.embed_scale.to(out.dtype)


def _unpack_ct_int(packed: torch.Tensor, num_bits: int, shape) -> torch.Tensor:
    """Unpack compressed-tensors ``pack-quantized`` int32 words into signed ints (CPU)."""
    try:  # compressed-tensors >= 0.20
        from compressed_tensors.compressors.pack_quantized.helpers import unpack_from_int32
    except ImportError:  # older layout
        from compressed_tensors.compressors.quantized_compressors.pack_quantized import unpack_from_int32

    return unpack_from_int32(packed, num_bits, torch.Size([int(s) for s in shape]))


class AirLLMGemma4(AirLLMBaseModel):
    """Gemma 4 dense / MoE / E-series, text-only."""

    def set_layer_names_dict(self):
        self.layer_names_dict = {
            "embed": "model.language_model.embed_tokens",
            "layer_prefix": "model.language_model.layers",
            "norm": "model.language_model.norm",
            # Gemma 4 ties lm_head to embed_tokens. The QAT "-ct" checkpoint ships a
            # separate 2-bit lm_head, which is then streamed like a layer.
            "lm_head": "lm_head",
            "resident": [
                "model.language_model.per_layer_model_projection",
                "model.language_model.per_layer_projection_norm",
            ],
            "cpu_resident": [PLE_NAME],
        }

    def init_model(self):
        # Text-only: drop the audio / vision towers from the config before the empty model is
        # built. Besides saving memory, Gemma4AudioModel has no SDPA attention, and its
        # presence would force AirLLM to fall back to eager attention for the whole model.
        for attr in ("audio_config", "vision_config"):
            if getattr(self.config, attr, None) is not None:
                setattr(self.config, attr, None)
        super().init_model()

    def move_layer_to_device(self, state_dict):
        # KV-shared layers (the last ``num_kv_shared_layers``) reuse earlier layers' K/V and have
        # no k_proj / v_proj modules, yet the checkpoint still ships those tensors (transformers
        # drops them via ``_keys_to_ignore_on_load_unexpected``). Skip tensors with no module.
        cache = self.__dict__.setdefault("_module_exists", {})
        keep = {}
        for key, value in state_dict.items():
            path = key.split(".4bit.")[0].split(".8bit.")[0].rsplit(".", 1)[0]
            if path not in cache:
                try:
                    self.model.get_submodule(path)
                    cache[path] = True
                except AttributeError:
                    cache[path] = False
            if cache[path]:
                keep[key] = value
        return super().move_layer_to_device(keep)

    def _install_streaming_hooks(self):
        text_cfg = self.config.get_text_config()
        tied = bool(getattr(self.config, "tie_word_embeddings", False))
        if not getattr(text_cfg, "hidden_size_per_layer_input", 0) or tied:
            return super()._install_streaming_hooks()
        # Gemma4Model.forward reads ``embed_tokens.weight[pad_token_id]`` directly (outside the
        # module call) for PLE models, so a streamed input embedding would still be on 'meta'
        # there. With tied embeddings AirLLM already keeps it resident; the untied QAT "-ct"
        # checkpoint needs the same treatment. lm_head is still streamed.
        self.tie_word_embeddings = False
        self.move_layer_to_device(self.load_layer_to_cpu(self.layer_names[0]))
        self._streamed_indices = list(range(1, len(self.layer_names)))
        self._streamed_set = set(self._streamed_indices)
        self._setup_expert_streaming()
        for idx in self._streamed_indices:
            module = self.layers[idx]
            module._airllm_idx = idx
            module.register_forward_pre_hook(self._pre_hook)
            module.register_forward_hook(self._post_hook)

    # ---- PLE table on CPU -----------------------------------------------------------------

    def _load_cpu_resident_modules(self):
        self._cpu_resident_params = set()
        try:
            state_dict = airllm_utils.load_merged_ngram_embedding(self.checkpoint_path, PLE_NAME)
        except FileNotFoundError:
            return  # dense Gemma 4 (12B, 26B-A4B) has no PLE
        module = self.model.get_submodule(PLE_NAME)

        if f"{PLE_NAME}.weight_packed" in state_dict:
            # compressed-tensors pack-quantized (QAT checkpoints): int4, symmetric, group-wise.
            scheme = getattr(module, "quantization_scheme", None)
            num_bits = scheme.weights.num_bits if scheme is not None else 4
            q = _unpack_ct_int(state_dict[f"{PLE_NAME}.weight_packed"], num_bits,
                               state_dict[f"{PLE_NAME}.weight_shape"].tolist())
            scale = state_dict[f"{PLE_NAME}.weight_scale"].float()
            group = q.shape[1] // scale.shape[1]
            table = torch.empty(q.shape, dtype=self.running_dtype)
            step = 8192  # dequantize in row chunks to bound peak host RAM
            for r in range(0, q.shape[0], step):
                qs = q[r:r + step].float().view(-1, scale.shape[1], group)
                table[r:r + step] = (qs * scale[r:r + step, :, None]).view(qs.shape[0], -1).to(self.running_dtype)
            del q, scale
        else:
            table = state_dict[f"{PLE_NAME}.weight"].to(self.running_dtype)

        module._airllm_cpu_table = table.contiguous()
        if not hasattr(module, "padding_idx"):
            module.padding_idx = None
        module.forward = types.MethodType(_ple_cpu_forward, module)
        # Marks the post-hook to release only streamed tensors instead of module.to('meta').
        self._cpu_resident_params.add(f"{PLE_NAME}.weight")
        print(f"[gemma4] PLE table kept on CPU: {tuple(table.shape)} {table.dtype} "
              f"({table.numel() * table.element_size() / 1e9:.2f} GB host RAM)")


In [ ]:
%%writefile /kaggle/working/airllm-benchmarks/bench/models.py
"""Model registry and AirLLM class selection."""

from __future__ import annotations

import airllm_compat  # noqa: F401  (patches AirLLM for transformers 5.2x)
from airllm import AirLLMQwen3_5, AutoModel

from airllm_gemma4 import AirLLMGemma4


class AirLLMQwen3_5TextOnly(AirLLMQwen3_5):
    """AirLLM's stock Qwen3.5/3.8 class keeps the vision tower resident on the GPU.

    These benchmarks are text-only, so the vision tower is not split or loaded. That keeps
    VRAM numbers comparable with the (text-only) Gemma 4 adapter. Use ``--stock`` to run
    AirLLM's unmodified class instead.
    """

    def set_layer_names_dict(self):
        super().set_layer_names_dict()
        self.layer_names_dict["resident"] = []


# short name -> (HF repo id, notes)
MODELS = {
    "gemma-4-E2B-it-qat-mobile-ct": "google/gemma-4-E2B-it-qat-mobile-ct",
    "gemma-4-E4B-it": "google/gemma-4-E4B-it",
    "gemma-4-12B-it": "google/gemma-4-12B-it",
    "gemma-4-26B-A4B-it": "google/gemma-4-26B-A4B-it",
    "Qwen3.5-9B": "Qwen/Qwen3.5-9B",
    "Qwen3.6-27B": "Qwen/Qwen3.6-27B",
    "Qwen3.8-27B": "Qwen/Qwen3.8-27B",
}


def airllm_class_for(repo_id: str, stock: bool = False):
    """Return the AirLLM class to use, or None to let ``AutoModel`` decide."""
    if "gemma-4" in repo_id.lower():
        return AirLLMGemma4  # no stock class exists in AirLLM 4.0.0
    if stock:
        return None
    if repo_id.startswith("Qwen/Qwen3."):
        return AirLLMQwen3_5TextOnly
    return None


def build(repo_id: str, stock: bool = False, **kwargs):
    cls = airllm_class_for(repo_id, stock)
    if cls is None:
        return AutoModel.from_pretrained(repo_id, **kwargs)
    return cls(repo_id, **kwargs)


In [ ]:
%%writefile /kaggle/working/airllm-benchmarks/bench/lazy_split.py
"""Layer-by-layer splitter that never loads a whole checkpoint shard.

AirLLM's ``split_and_save_layers`` reads each shard with ``safetensors.torch.load_file``, i.e. the
whole shard at once. ``google/gemma-4-26B-A4B-it`` ships a single 49.9 GB shard, which fails
on a 31 GB-RAM machine ("unable to mmap 49907246508 bytes ... Cannot allocate memory"). And when
AirLLM streams shards from the Hub it has to hold that whole file on disk, which overflowed Kaggle's
scratch quota.

This splitter parses the safetensors header and reads only the byte ranges of one layer at a
time, either from a locally cached shard or directly from the Hub over HTTP range requests
(``HfFileSystem``), so neither the shard nor the checkpoint ever has to fit in RAM or on disk.
It writes exactly the files AirLLM expects (``<layer>.safetensors`` + ``.done`` markers in
``splitted_model[.4bit]``), compressing each layer with AirLLM's own (patched) compressor, so
AirLLM finds the split and skips its own splitter.
"""

from __future__ import annotations

import json
import struct
import time
from collections import defaultdict
from pathlib import Path

import torch

_DTYPES = {
    "BF16": torch.bfloat16, "F16": torch.float16, "F32": torch.float32, "F64": torch.float64,
    "I8": torch.int8, "U8": torch.uint8, "I16": torch.int16, "I32": torch.int32, "I64": torch.int64,
    "BOOL": torch.bool, "F8_E4M3": torch.float8_e4m3fn, "F8_E5M2": torch.float8_e5m2,
}


def _read_header(f):
    (n,) = struct.unpack("<Q", f.read(8))
    header = json.loads(f.read(n))
    header.pop("__metadata__", None)
    return header, 8 + n


def _open_shard(repo_id, filename, revision=None):
    """Local cached file if present, else a random-access HTTP file on the Hub."""
    from huggingface_hub import hf_hub_download

    try:
        path = hf_hub_download(repo_id, filename, revision=revision, local_files_only=True)
        return open(path, "rb"), "local"
    except Exception:  # noqa: BLE001 - not cached
        from huggingface_hub import HfFileSystem

        fs = HfFileSystem()
        rev = f"@{revision}" if revision else ""
        return fs.open(f"{repo_id}{rev}/{filename}", "rb", block_size=0), "remote"


def _iter_ranges(f, where, data_start, items, repo_id, filename, revision=None, workers=8):
    """Yield (key, meta, bytes) in order. Remote: ``workers`` concurrent range requests
    (one HTTP stream to the Hub tops out at a few MB/s); at most ``workers`` tensors in flight."""
    if where == "local":
        for key, meta in items:
            start, end = meta["data_offsets"]
            f.seek(data_start + start)
            yield key, meta, bytearray(f.read(end - start)) if end > start else bytearray()
        return

    from collections import deque
    from concurrent.futures import ThreadPoolExecutor

    from huggingface_hub import HfFileSystem

    fs = HfFileSystem()
    path = f"{repo_id}{'@' + revision if revision else ''}/{filename}"

    def fetch(meta):
        start, end = meta["data_offsets"]
        if end <= start:
            return bytearray()
        for attempt in range(5):
            try:
                return bytearray(fs.cat_file(path, start=data_start + start, end=data_start + end))
            except Exception:  # noqa: BLE001 - transient network errors
                if attempt == 4:
                    raise
                time.sleep(2 ** attempt)

    with ThreadPoolExecutor(workers) as pool:
        window = deque()
        it = iter(items)
        for key, meta in it:
            window.append((key, meta, pool.submit(fetch, meta)))
            if len(window) >= workers:
                break
        while window:
            key, meta, fut = window.popleft()
            nxt = next(it, None)
            if nxt is not None:
                window.append((nxt[0], nxt[1], pool.submit(fetch, nxt[1])))
            yield key, meta, fut.result()


def _layer_list(layer_names, keys):
    from airllm.utils import cpu_resident_module_names, layer_owner

    prefix = layer_names["layer_prefix"]
    n_layers = len({int(k[len(prefix):].split(".")[1]) for k in keys if prefix in k})
    layers = [layer_names["embed"]] + [f"{prefix}.{i}" for i in range(n_layers)] + \
             [layer_names["norm"], layer_names["lm_head"]]
    if "rotary_pos_emb" in layer_names:
        layers = [layer_names["rotary_pos_emb"]] + layers
    layers += list(layer_names.get("resident", []))
    layers += cpu_resident_module_names(layer_names, keys)
    layers = [l + "." for l in layers]
    layers = [l for l in layers if any(k.startswith(l) for k in keys)]
    owned = {layer_owner(k, layers) for k in keys}
    return [l for l in layers if l in owned], layer_owner


def lazy_split(repo_id, layers_root, layer_names, compression=None, revision=None, log=print):
    """Split ``repo_id`` into AirLLM per-layer files under ``layers_root``. Returns stats."""
    import airllm.utils as airllm_utils
    from airllm.persist import ModelPersister
    from huggingface_hub import hf_hub_download

    saving_path = Path(layers_root) / ("splitted_model" + (f".{compression}" if compression else ""))
    saving_path.mkdir(parents=True, exist_ok=True)
    persister = ModelPersister.get_model_persister()

    index_file = hf_hub_download(repo_id, "model.safetensors.index.json", revision=revision)
    weight_map = json.load(open(index_file))["weight_map"]
    layers, layer_owner = _layer_list(layer_names, list(weight_map))
    todo = [l for l in layers if not persister.model_persist_exist(l, saving_path)]
    if not todo:
        log(f"[lazy_split] all {len(layers)} layer files already present in {saving_path}")
        return {"layers": len(layers), "bytes_read": 0, "sources": []}

    expected = defaultdict(int)  # layer -> number of tensors it owns
    owner_of = {}
    for k in weight_map:
        o = layer_owner(k, layers)
        if o is not None:
            owner_of[k] = o
            expected[o] += 1

    pending = defaultdict(dict)
    bytes_read, sources = 0, []
    t0 = time.time()
    for shard in sorted(set(weight_map.values())):
        f, where = _open_shard(repo_id, shard, revision)
        sources.append(where)
        header, data_start = _read_header(f)
        items = sorted(((k, v) for k, v in header.items() if owner_of.get(k) in todo),
                       key=lambda kv: kv[1]["data_offsets"][0])
        log(f"[lazy_split] {shard} ({where}): {len(items)} tensors for {len({owner_of[k] for k, _ in items})} layers")
        for key, meta, buf in _iter_ranges(f, where, data_start, items, repo_id, shard, revision):
            start, end = meta["data_offsets"]
            bytes_read += end - start
            dtype = _DTYPES[meta["dtype"]]
            t = torch.frombuffer(buf, dtype=dtype) if buf else torch.empty(0, dtype=dtype)
            layer = owner_of[key]
            pending[layer][key] = t.reshape(meta["shape"])
            if len(pending[layer]) == expected[layer]:
                state = airllm_utils.compress_layer_state_dict(pending.pop(layer), compression)
                persister.persist_model(state, layer, saving_path)
                del state
        f.close()
    if pending:
        raise RuntimeError(f"incomplete layers after reading all shards: {sorted(pending)[:5]}")
    dt = time.time() - t0
    log(f"[lazy_split] wrote {len(todo)} layer files, read {bytes_read / 1e9:.1f} GB in {dt:.0f} s "
        f"({bytes_read / 1e9 / max(dt, 1e-9):.2f} GB/s)")
    return {"layers": len(layers), "bytes_read": bytes_read, "seconds": dt, "sources": sorted(set(sources))}


In [ ]:
%%writefile /kaggle/working/airllm-benchmarks/bench/run_bench.py
#!/usr/bin/env python
"""Benchmark one model under AirLLM and write a JSON result.

Phases (each timed separately):
  download  - huggingface snapshot of the original checkpoint (0 if cached)
  convert   - AirLLM's one-time split into per-layer shards (+ optional 4/8-bit compression)
  init      - build the empty model and load resident modules
  generate  - greedy generation, run twice:
                cold: split files evicted from the OS page cache first (posix_fadvise)
                warm: immediately after, whatever the page cache kept
Per-token timestamps come from a LogitsProcessor, so prefill (TTFT) and decode speed are
measured separately. Layer I/O time and bytes are taken by wrapping AirLLM's loader.
"""

from __future__ import annotations

import argparse
import json
import os
import platform
import socket
import sys
import threading
import time
import traceback
from pathlib import Path

import psutil
import torch

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))

DEFAULT_PROMPT = "Explain how a CPU cache works in one short paragraph."


# ---------------------------------------------------------------------------------------------
# monitoring


class Sampler(threading.Thread):
    """Samples process RSS and GPU memory (NVML, whole device and this process) every 100 ms."""

    def __init__(self, interval=0.1):
        super().__init__(daemon=True)
        self.interval = interval
        self.proc = psutil.Process()
        self._stop = threading.Event()
        self.reset()
        try:
            import pynvml

            pynvml.nvmlInit()
            self.nvml = pynvml
            self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        except Exception:  # noqa: BLE001
            self.nvml = None

    def reset(self):
        self.peak_rss = 0
        self.peak_gpu_device = 0
        self.peak_gpu_process = 0
        self.other_gpu_pids = set()

    def gpu(self):
        if self.nvml is None:
            return 0, 0
        dev = self.nvml.nvmlDeviceGetMemoryInfo(self.handle).used
        mine = 0
        try:
            for p in self.nvml.nvmlDeviceGetComputeRunningProcesses(self.handle):
                if p.pid == os.getpid():
                    mine = p.usedGpuMemory or 0
                elif (p.usedGpuMemory or 0) > 256 * 2**20:
                    self.other_gpu_pids.add(p.pid)  # another workload shared the GPU
        except Exception:  # noqa: BLE001
            pass
        return dev, mine

    def run(self):
        while not self._stop.is_set():
            try:
                self.peak_rss = max(self.peak_rss, self.proc.memory_info().rss)
                dev, mine = self.gpu()
                self.peak_gpu_device = max(self.peak_gpu_device, dev)
                self.peak_gpu_process = max(self.peak_gpu_process, mine)
            except Exception:  # noqa: BLE001
                pass
            time.sleep(self.interval)

    def stop(self):
        self._stop.set()

    def snapshot(self):
        return {
            "peak_rss_gb": round(self.peak_rss / 1e9, 3),
            "peak_gpu_device_used_gb": round(self.peak_gpu_device / 1e9, 3),
            "peak_gpu_process_gb": round(self.peak_gpu_process / 1e9, 3),
            "other_gpu_pids_seen": sorted(self.other_gpu_pids),
        }


class IOStats:
    def __init__(self):
        self.lock = threading.Lock()
        self.reset()

    def reset(self):
        self.load_s = 0.0
        self.load_bytes = 0
        self.load_calls = 0
        self.to_device_s = 0.0

    def as_dict(self):
        return {
            "layer_load_s": round(self.load_s, 3),
            "layer_load_gb": round(self.load_bytes / 1e9, 3),
            "layer_load_calls": self.load_calls,
            "layer_load_gbps": round(self.load_bytes / 1e9 / self.load_s, 3) if self.load_s else None,
            "to_device_s": round(self.to_device_s, 3),
        }


def instrument(model, stats: IOStats):
    """Wrap AirLLM's per-layer loader (runs on the prefetch thread too) and device move."""
    orig_load = model.load_layer_to_cpu
    orig_move = model.move_layer_to_device
    ckpt = Path(model.checkpoint_path)

    def load(layer_name):
        t = time.perf_counter()
        out = orig_load(layer_name)
        dt = time.perf_counter() - t
        size = 0
        f = ckpt / (layer_name + ".safetensors")
        if f.exists():
            size = f.stat().st_size
        with stats.lock:
            stats.load_s += dt
            stats.load_bytes += size
            stats.load_calls += 1
        return out

    def move(state_dict):
        t = time.perf_counter()
        out = orig_move(state_dict)
        torch.cuda.synchronize()
        with stats.lock:
            stats.to_device_s += time.perf_counter() - t
        return out

    model.load_layer_to_cpu = load
    model.move_layer_to_device = move


def evict_page_cache(path: Path) -> int:
    """Drop clean cached pages of every file under ``path`` (no root needed)."""
    n = 0
    for f in path.rglob("*"):
        if f.is_file():
            try:
                fd = os.open(f, os.O_RDONLY)
                os.fdatasync(fd)
                os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
                os.close(fd)
                n += 1
            except OSError:
                pass
    return n


def dir_size_gb(path: Path) -> float:
    if not path.exists():
        return 0.0
    return round(sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e9, 3)


def env_info():
    import airllm  # noqa: F401
    import transformers
    from importlib.metadata import version

    info = {
        "hostname_tag": os.environ.get("BENCH_HOST_TAG", socket.gethostname()),
        "platform": platform.platform(),
        "python": platform.python_version(),
        "cpu": platform.processor() or "",
        "cpu_count": psutil.cpu_count(),
        "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 1),
        "torch": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "transformers": transformers.__version__,
        "airllm": version("airllm"),
        "bitsandbytes": version("bitsandbytes"),
    }
    try:
        info["compressed_tensors"] = version("compressed-tensors")
    except Exception:  # noqa: BLE001
        pass
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info.update({
            "gpu": p.name,
            "gpu_vram_gb": round(p.total_memory / 1e9, 2),
            "gpu_cc": f"{p.major}.{p.minor}",
            "gpu_count": torch.cuda.device_count(),
            "bf16_native": torch.cuda.is_bf16_supported(including_emulation=False),
        })
    try:
        with open("/proc/cpuinfo") as fh:
            for line in fh:
                if line.startswith("model name"):
                    info["cpu"] = line.split(":", 1)[1].strip()
                    break
    except OSError:
        pass
    return info


# ---------------------------------------------------------------------------------------------
# generation


class TokenTimer:
    """LogitsProcessor that timestamps each decoding step (called once per generated token)."""

    def __init__(self):
        self.times = []

    def __call__(self, input_ids, scores):
        torch.cuda.synchronize()
        self.times.append(time.perf_counter())
        return scores


def build_inputs(tokenizer, prompt):
    messages = [{"role": "user", "content": prompt}]
    text = None
    if getattr(tokenizer, "chat_template", None):
        for kw in ({"enable_thinking": False}, {}):
            try:
                text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **kw)
                break
            except Exception:  # noqa: BLE001
                continue
    used_template = text is not None
    if text is None:
        text = prompt
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=not used_template)
    return enc, used_template, text


def timed_generate(model, enc, max_new_tokens, sampler, stats):
    from transformers import LogitsProcessorList

    timer = TokenTimer()
    stats.reset()
    sampler.reset()
    torch.cuda.reset_peak_memory_stats()
    input_ids = enc["input_ids"].cuda()
    attn = enc.get("attention_mask")
    kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
        use_cache=True,
        return_dict_in_generate=True,
        logits_processor=LogitsProcessorList([timer]),
    )
    if attn is not None:
        kwargs["attention_mask"] = attn.cuda()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = model.generate(input_ids, **kwargs)
    torch.cuda.synchronize()
    t_end = time.perf_counter()

    new_tokens = out.sequences[0, input_ids.shape[1]:]
    ts = timer.times
    n = len(ts)
    ttft = ts[0] - t0 if n else None
    decode_s = ts[-1] - ts[0] if n > 1 else None
    per_tok = [round(b - a, 4) for a, b in zip(ts, ts[1:])]
    res = {
        "prompt_tokens": int(input_ids.shape[1]),
        "new_tokens": int(new_tokens.shape[0]),
        "total_s": round(t_end - t0, 3),
        "ttft_s": round(ttft, 3) if ttft is not None else None,
        "decode_s": round(decode_s, 3) if decode_s is not None else None,
        "decode_tok_per_s": round((n - 1) / decode_s, 4) if decode_s else None,
        "s_per_decode_token": round(decode_s / (n - 1), 3) if decode_s else None,
        "end_to_end_tok_per_s": round(n / (t_end - t0), 4) if n else None,
        "per_token_s": per_tok,
        "torch_peak_allocated_gb": round(torch.cuda.max_memory_allocated() / 1e9, 3),
        "torch_peak_reserved_gb": round(torch.cuda.max_memory_reserved() / 1e9, 3),
        "text": model.tokenizer.decode(new_tokens, skip_special_tokens=True),
    }
    res.update(stats.as_dict())
    res.update(sampler.snapshot())
    return res


# ---------------------------------------------------------------------------------------------


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, help="short name from bench/models.py or a HF repo id")
    ap.add_argument("--compression", default="none", choices=["none", "4bit", "8bit", "4bit-2d", "8bit-2d"],
                    help="'-2d': compress only weight matrices (>= 2 dims), keep norms/scalars in bf16")
    ap.add_argument("--dtype", default="auto", choices=["auto", "fp16", "bf16"])
    ap.add_argument("--max-new-tokens", type=int, default=32)
    ap.add_argument("--prompt", default=DEFAULT_PROMPT)
    ap.add_argument("--work-dir", default=os.environ.get("AIRLLM_WORK", str(Path.home() / "airllm-work")))
    ap.add_argument("--out-dir", default=str(HERE.parent / "results" / "local"))
    ap.add_argument("--delete-original", action="store_true",
                    help="let AirLLM delete original shards after splitting (needs an index.json)")
    ap.add_argument("--stream-download", action="store_true",
                    help="skip the up-front download; AirLLM fetches shards one by one while splitting "
                         "(with --delete-original the full checkpoint never sits on disk)")
    ap.add_argument("--no-prefetch", action="store_true")
    ap.add_argument("--stock", action="store_true", help="use AirLLM's stock class (Qwen keeps vision tower)")
    ap.add_argument("--no-gc", action="store_true", help="skip AirLLM's per-layer gc.collect()")
    ap.add_argument("--lazy-split", action="store_true",
                    help="split with bench/lazy_split.py (byte-range reads, local or straight from the Hub) "
                         "instead of AirLLM's whole-shard splitter")
    ap.add_argument("--skip-warm", action="store_true")
    ap.add_argument("--tag", default="", help="suffix for the result file name")
    args = ap.parse_args()

    matrices_only = args.compression.endswith("-2d")
    if matrices_only:
        os.environ["AIRLLM_COMPRESS_MIN_DIM"] = "2"  # read by airllm_compat at import
    if args.no_gc:
        os.environ["AIRLLM_NO_GC"] = "1"
    from models import MODELS, airllm_class_for, build

    repo_id = MODELS.get(args.model, args.model)
    short = args.model if args.model in MODELS else repo_id.split("/")[-1]
    compression = None if args.compression == "none" else args.compression.replace("-2d", "")
    dtype = {"auto": None, "fp16": torch.float16, "bf16": torch.bfloat16}[args.dtype]

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    name = f"{short}__{args.compression}" + ("__nogc" if args.no_gc else "") + \
           (f"__{args.dtype}" if args.dtype != "auto" else "") + \
           (f"__{args.tag}" if args.tag else "")
    out_file = out_dir / f"{name}.json"

    layers_root = Path(args.work_dir) / "layers" / (short + ("-2d" if matrices_only else ""))
    split_dir = layers_root / ("splitted_model" + (f".{compression}" if compression else ""))

    result = {
        "model": short,
        "repo_id": repo_id,
        "compression": args.compression,
        "dtype_arg": args.dtype,
        "no_gc": bool(args.no_gc),
        "max_new_tokens": args.max_new_tokens,
        "prompt": args.prompt,
        "airllm_class": None,
        "status": "running",
        "started": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "env": env_info(),
        "phases": {},
        "runs": {},
    }

    def save():
        out_file.write_text(json.dumps(result, indent=2))

    sampler = Sampler()
    sampler.start()
    stats = IOStats()
    phase = "download"
    try:
        import huggingface_hub
        from huggingface_hub import snapshot_download
        from airllm.utils import find_or_create_local_splitted_path

        cls = airllm_class_for(repo_id, args.stock)
        result["airllm_class"] = cls.__name__ if cls else "AutoModel"
        print(f"=== {repo_id} | compression={args.compression} | class={result['airllm_class']}", flush=True)

        # -- download --------------------------------------------------------------------------
        split_preexisting = split_dir.exists() and any(split_dir.glob("*.done"))
        t = time.perf_counter()
        if not split_preexisting and args.lazy_split:
            snapshot_download(repo_id, ignore_patterns=["*.safetensors", "*.bin", "*.pth"])  # metadata only
        elif not split_preexisting and not args.stream_download:
            snapshot_download(repo_id)
        result["phases"]["stream_download"] = bool(args.stream_download)
        result["phases"]["download_s"] = round(time.perf_counter() - t, 2)

        # -- convert (split) ----------------------------------------------------------------------
        phase = "convert"
        sampler.reset()
        if cls is None:
            from airllm.auto_model import AutoModel as _AM
            import importlib
            mod, cname = _AM.get_module_class(repo_id)
            cls_for_names = getattr(importlib.import_module(mod), cname)
        else:
            cls_for_names = cls
        probe = cls_for_names.__new__(cls_for_names)
        probe.set_layer_names_dict()
        t = time.perf_counter()
        if args.lazy_split:
            from lazy_split import lazy_split

            result["phases"]["lazy_split"] = lazy_split(repo_id, layers_root, probe.layer_names_dict,
                                                        compression=compression)
        find_or_create_local_splitted_path(repo_id, str(layers_root), compression=compression,
                                           layer_names=probe.layer_names_dict,
                                           delete_original=args.delete_original)
        result["phases"]["convert_s"] = round(time.perf_counter() - t, 2)
        result["phases"]["split_preexisting"] = split_preexisting
        result["phases"]["split_size_gb"] = dir_size_gb(split_dir)
        result["phases"]["convert_mem"] = sampler.snapshot()
        save()
        # all shards are split now (stream mode needed the network until here)
        huggingface_hub.constants.HF_HUB_OFFLINE = True

        # -- init ---------------------------------------------------------------------------------
        phase = "init"
        sampler.reset()
        torch.cuda.reset_peak_memory_stats()
        t = time.perf_counter()
        model = build(repo_id, stock=args.stock, layer_shards_saving_path=str(layers_root),
                      compression=compression, dtype=dtype, prefetching=not args.no_prefetch,
                      delete_original=args.delete_original)
        torch.cuda.synchronize()
        result["phases"]["init_s"] = round(time.perf_counter() - t, 2)
        result["phases"]["init_mem"] = sampler.snapshot()
        result["phases"]["init_torch_allocated_gb"] = round(torch.cuda.memory_allocated() / 1e9, 3)
        result["running_dtype"] = str(model.running_dtype)
        result["prefetching"] = bool(model.prefetching)
        result["n_streamed_modules"] = len(getattr(model, "_streamed_indices", []))
        instrument(model, stats)
        save()

        enc, used_template, _ = build_inputs(model.tokenizer, args.prompt)
        result["chat_template"] = used_template

        # -- generate -----------------------------------------------------------------------------
        phase = "generate_cold"
        n_evicted = evict_page_cache(split_dir)
        print(f"evicted {n_evicted} split files from the page cache", flush=True)
        result["runs"]["cold"] = timed_generate(model, enc, args.max_new_tokens, sampler, stats)
        print(json.dumps({k: v for k, v in result["runs"]["cold"].items() if k != "per_token_s"}, indent=1), flush=True)
        save()
        if not args.skip_warm:
            phase = "generate_warm"
            result["runs"]["warm"] = timed_generate(model, enc, args.max_new_tokens, sampler, stats)
            print(json.dumps({k: v for k, v in result["runs"]["warm"].items() if k != "per_token_s"}, indent=1), flush=True)
        result["status"] = "ok"
    except BaseException as e:  # noqa: BLE001 - record every failure, including OOM / Ctrl-C
        result["status"] = "error"
        result["error_phase"] = phase
        result["error"] = f"{type(e).__name__}: {e}"[:2000]
        result["traceback"] = traceback.format_exc()[-6000:]
        print(result["traceback"], file=sys.stderr, flush=True)
    finally:
        sampler.stop()
        result["finished"] = time.strftime("%Y-%m-%dT%H:%M:%S")
        save()
        print(f"wrote {out_file} status={result['status']}", flush=True)
    sys.exit(0 if result["status"] == "ok" else 1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/airllm-benchmarks/bench/run_matrix.py
#!/usr/bin/env python
"""Run the whole benchmark matrix sequentially, one model at a time.

* each (model, compression) runs in its own process, so memory peaks are not shared
* waits until no other process is using the GPU before each job (shared workstation)
* prefetches the next model's checkpoint in the background when disk space allows
* deletes a model's split layers and HF cache once all its jobs are done (--cleanup)
* resumable: a job whose JSON already has status "ok" is skipped
"""

from __future__ import annotations

import argparse
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

HERE = Path(__file__).resolve().parent
ROOT = HERE.parent
PY = sys.executable

# original checkpoint size in GB (safetensors), used for disk planning
SIZE_GB = {
    "gemma-4-E2B-it-qat-mobile-ct": 2.7,
    "gemma-4-E4B-it": 16.0,
    "Qwen3.5-9B": 19.3,
    "gemma-4-12B-it": 23.9,
    "gemma-4-26B-A4B-it": 51.6,
    "Qwen3.6-27B": 55.6,
    "Qwen3.8-27B": 55.6,
}

# (model, [compressions], max_new_tokens, extra run_bench args)
PLANS = {
    # 8 GB laptop GPU, ~150 GB free disk: keep originals, run bf16 and 4-bit for every model
    # "<compression>[:nogc]": stock AirLLM ("none" = bf16, "4bit" = NF4 on every tensor) plus a
    # tuned variant: NF4 on weight matrices only and no per-layer gc.collect()
    "local": [
        ("gemma-4-E2B-it-qat-mobile-ct", ["none", "none:nogc"], 32, []),  # already int2/4/8 (compressed-tensors)
        ("gemma-4-E4B-it", ["4bit", "none", "4bit-2d:nogc", "none:nogc"], 32, []),
        ("Qwen3.5-9B", ["4bit", "none", "4bit-2d:nogc", "none:nogc"], 32, []),
        ("gemma-4-12B-it", ["4bit", "none", "4bit-2d:nogc", "none:nogc"], 32, []),
        # one 49.9 GB shard: AirLLM's splitter can't load it with 31 GB RAM -> lazy byte-range splitter
        ("gemma-4-26B-A4B-it", ["4bit", "none", "4bit-2d:nogc"], 16, ["--lazy-split"]),
        ("Qwen3.6-27B", ["4bit", "none", "4bit-2d:nogc"], 16, []),
        ("Qwen3.8-27B", ["4bit", "none", "4bit-2d:nogc"], 16, []),
    ],
    # Kaggle T4: limited disk, so the >25B models stream shards and keep only the 4-bit split
    "kaggle": [
        ("gemma-4-E2B-it-qat-mobile-ct", ["none"], 32, []),
        ("gemma-4-E4B-it", ["4bit", "none"], 32, []),
        ("Qwen3.5-9B", ["4bit", "none"], 32, []),
        ("gemma-4-12B-it", ["4bit", "none"], 32, []),
        ("gemma-4-26B-A4B-it", ["4bit"], 16, ["--stream-download", "--delete-original"]),
        ("Qwen3.6-27B", ["4bit"], 16, ["--stream-download", "--delete-original"]),
        ("Qwen3.8-27B", ["4bit"], 16, ["--stream-download", "--delete-original"]),
    ],
    # Kaggle run 2: v1 lost the >25B models to the scratch-disk quota (streaming a 49.9 GB shard).
    # Split them straight from the Hub (byte ranges; no checkpoint on disk) and add tuned variants.
    "kaggle2": [
        # Qwen 27B shards are ~3.7 GB, so AirLLM's own shard streaming fits the quota
        ("Qwen3.6-27B", ["4bit", "4bit-2d:nogc"], 16, ["--stream-download", "--delete-original"]),
        ("Qwen3.8-27B", ["4bit", "4bit-2d:nogc"], 16, ["--stream-download", "--delete-original"]),
        ("gemma-4-E2B-it-qat-mobile-ct", ["none:nogc"], 32, []),
        ("gemma-4-E4B-it", ["4bit-2d:nogc"], 32, []),
        ("Qwen3.5-9B", ["4bit-2d:nogc"], 32, []),
        ("gemma-4-12B-it", ["4bit-2d:nogc"], 32, []),
        # 49.9 GB single shard: split straight from the Hub by byte ranges (never on disk)
        ("gemma-4-26B-A4B-it", ["4bit-2d:nogc"], 16, ["--lazy-split"]),
    ],
}


def repo_of(model):
    sys.path.insert(0, str(HERE))
    from models import MODELS

    return MODELS[model]


def hf_cache_dir(model):
    from huggingface_hub import constants  # honours HF_HOME / HF_HUB_CACHE

    return Path(constants.HF_HUB_CACHE) / ("models--" + repo_of(model).replace("/", "--"))


def free_gb(path="/"):
    return shutil.disk_usage(path).free / 1e9


def other_gpu_procs():
    """PIDs (other than ours) with a CUDA context, and device memory used in GB."""
    try:
        import pynvml

        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        # ignore tiny desktop clients (e.g. gnome-remote-desktop holds ~100 MB)
        procs = [p.pid for p in pynvml.nvmlDeviceGetComputeRunningProcesses(h)
                 if p.pid != os.getpid() and (p.usedGpuMemory or 0) > 256 * 2**20]
        used = pynvml.nvmlDeviceGetMemoryInfo(h).used / 1e9
        return procs, used
    except Exception:  # noqa: BLE001
        return [], 0.0


def wait_for_idle_gpu(max_used_gb=1.0, poll_s=30, idle_for_s=300):
    """Block until no other process has used the GPU for ``idle_for_s`` seconds in a row
    (another workload on this machine runs sweeps back to back with short gaps)."""
    waited = 0
    idle_since = None
    while True:
        procs, used = other_gpu_procs()
        if not procs and used < max_used_gb:
            idle_since = idle_since if idle_since is not None else waited
            if waited - idle_since >= idle_for_s or waited == 0 and idle_for_s == 0:
                if waited:
                    print(f"[matrix] GPU idle for {idle_for_s // 60} min (waited {waited // 60} min total)", flush=True)
                return
        else:
            idle_since = None
        if waited % 900 == 0 and procs:
            print(f"[matrix] GPU busy (pids={procs}, used={used:.1f} GB); waiting...", flush=True)
        time.sleep(poll_s)
        waited += poll_s


def result_ok(out_dir, model, comp):
    f = out_dir / f"{model}__{comp.replace(':', '__')}.json"
    if not f.exists():
        return False
    try:
        return json.loads(f.read_text()).get("status") == "ok"
    except Exception:  # noqa: BLE001
        return False


def prefetch(model):
    log = open(ROOT / "logs" / f"prefetch_{model}.log", "w")
    hf = Path(PY).with_name("hf")
    cmd = [str(hf), "download", repo_of(model), "--quiet"] if hf.exists() else \
        [PY, "-c", f"from huggingface_hub import snapshot_download; snapshot_download('{repo_of(model)}')"]
    print(f"[matrix] prefetching {model} in background", flush=True)
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out-dir", default=str(ROOT / "results" / "local-rtx2070"))
    ap.add_argument("--work-dir", default=str(Path.home() / "airllm-work"))
    ap.add_argument("--plan", default="local", choices=sorted(PLANS))
    ap.add_argument("--only", nargs="*", help="subset of model names")
    ap.add_argument("--cleanup", action="store_true", help="delete split layers + HF cache after each model")
    ap.add_argument("--no-prefetch", action="store_true")
    ap.add_argument("--no-wait-gpu", action="store_true")
    ap.add_argument("--timeout-min", type=int, default=240)
    args = ap.parse_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    (ROOT / "logs").mkdir(exist_ok=True)
    plan = [p for p in PLANS[args.plan] if not args.only or p[0] in args.only]
    env = dict(os.environ, BENCH_HOST_TAG=os.environ.get("BENCH_HOST_TAG", out_dir.name),
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")

    prefetcher = None
    for i, (model, comps, max_new, extra) in enumerate(plan):
        # background-download the next model if it fits alongside this one's peak
        if not args.no_prefetch and i + 1 < len(plan) and prefetcher is None:
            nxt = plan[i + 1][0]
            need_now = SIZE_GB[model] * (1 + 0.3 + 1) if not hf_cache_dir(model).exists() else SIZE_GB[model] * 1.3
            if free_gb() - need_now - SIZE_GB[nxt] > 15 and not all(result_ok(out_dir, nxt, c) for c in plan[i + 1][1]):
                prefetcher = prefetch(nxt)

        for comp in comps:
            if result_ok(out_dir, model, comp):
                print(f"[matrix] skip {model} {comp} (done)", flush=True)
                continue
            if not args.no_wait_gpu:
                wait_for_idle_gpu()
            print(f"[matrix] >>> {model} {comp}  free disk {free_gb():.0f} GB", flush=True)
            comp_arg, *flags = comp.split(":")
            cmd = [PY, str(HERE / "run_bench.py"), "--model", model, "--compression", comp_arg,
                   *(["--no-gc"] if "nogc" in flags else []),
                   "--max-new-tokens", str(max_new), "--out-dir", str(out_dir), "--work-dir", args.work_dir,
                   *extra]
            with open(ROOT / "logs" / f"{out_dir.name}_{model}__{comp.replace(':', '__')}.log", "w") as log:
                try:
                    rc = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, env=env,
                                        timeout=args.timeout_min * 60).returncode
                except subprocess.TimeoutExpired:
                    rc = "timeout"
            print(f"[matrix] <<< {model} {comp} rc={rc}", flush=True)

        if prefetcher is not None and prefetcher.poll() is not None:
            prefetcher = None

        if args.cleanup:
            shutil.rmtree(Path(args.work_dir) / "layers" / model, ignore_errors=True)
            shutil.rmtree(Path(args.work_dir) / "layers" / (model + "-2d"), ignore_errors=True)
            # don't delete a cache that the prefetcher is still writing for the *next* model
            shutil.rmtree(hf_cache_dir(model), ignore_errors=True)
            print(f"[matrix] cleaned {model}; free disk {free_gb():.0f} GB", flush=True)

    if prefetcher is not None:
        prefetcher.wait()
    print("[matrix] done", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
# disk / read speed of the scratch disk used for the split layers
!dd if=/dev/zero of=/tmp/ddtest bs=64M count=32 oflag=direct 2>&1 | tail -1
!dd if=/tmp/ddtest of=/dev/null bs=64M iflag=direct 2>&1 | tail -1
!rm -f /tmp/ddtest

In [ ]:
%cd /kaggle/working/airllm-benchmarks
import os, shutil
# keep checkpoints and split layers off /kaggle/working (20 GB output cap): use the roomiest scratch dir
cands = [c for c in ['/kaggle/tmp', '/tmp', '/root'] if os.path.isdir(c)]
SCRATCH = max(cands, key=lambda c: shutil.disk_usage(c).free)
print('scratch:', SCRATCH, round(shutil.disk_usage(SCRATCH).free / 1e9), 'GB free')
os.environ['HF_HOME'] = f'{SCRATCH}/hf'
os.environ['BENCH_HOST_TAG'] = 'kaggle-t4'
!python bench/run_matrix.py --plan kaggle2 --out-dir results/kaggle-t4 --work-dir {SCRATCH}/airllm-work --cleanup --no-prefetch --no-wait-gpu --timeout-min 300

In [ ]:
import json, glob
for f in sorted(glob.glob('/kaggle/working/airllm-benchmarks/results/kaggle-t4/*.json')):
    r = json.load(open(f))
    c, w = r['runs'].get('cold', {}), r['runs'].get('warm', {})
    print(f"{r['model']:30s} {r['compression']:5s} {r['status']:5s} "
          f"cold {c.get('decode_tok_per_s')} tok/s ttft {c.get('ttft_s')}s | "
          f"warm {w.get('decode_tok_per_s')} tok/s | vram {c.get('peak_gpu_process_gb')} GB "
          f"| {r.get('error', '')[:120]}")

In [ ]:
# full JSON results (read from the version log; same files are in the notebook output)
import json, glob
for f in sorted(glob.glob('/kaggle/working/airllm-benchmarks/results/kaggle-t4/*.json')):
    print('@@@RESULT', f.split('/')[-1])
    print(json.dumps(json.load(open(f))))
    print('@@@END')

In [ ]:
!tail -n 30 /kaggle/working/airllm-benchmarks/logs/*.log